# Глава 9. Алгоритмическая оптимизация
В этой главе поговорим про сущетсвующие модификации трансформерных моделей, применяющиеся для ускорения обучения и инференса. Акцент именно на алгоритмические хаки, про ускорение с помощью железа и параллелизации будем говорить в следующих главах. Обзор на методы борьбы с квадратичным вниманием, поговорим про FlashAttention и затронем спекулятивный инференс.

## Введение
В 4 главе мы отмечали, что при всех плюсах у трансформерной разитектуры есть два важных недостатка

1) Самовнимание считает попарные взаимодействия всех токенов, то есть его стоимость по вычислениям и памяти растёт как $O(n^2)$ по длине последовательности $n$. Пока контекст короткий, это незаметно, но на больших контекстах (длинных документах, репозиториях кода и многошаговых агентских диалогах) квадратичная сложность становится существенным ограничением, проявляющимся на этапе обучения и на префилле

2) Во время инференса токены порождаются по одному, каждый требует отдельного прохода модели, и на каждом шаге из медленной памяти (HBM) приходится заново вычитывать веса модели и весь накопленный KV-кэш. Здесь упираемся в пропускную способность памяти: тензорные ядра простаивают, GPU большую часть времени ждёт данные. Размер KV-кэша при этом линейно растёт с длиной контекста и размером батча

## Эффективный Attention
В 2020 году было несколько попыток победить квадратичную сложность. Базовая идея у всех общая: матрица внимания n×n на практике почти везде «лишняя», потому что softmax концентрирует вес на небольшом числе ключей, а нужные связи часто либо локальны, либо низкоранговы. Различаются методы тем, как именно они сокращают эту матрицу

### Longformer
[[Beltagi et al, 2020]](https://arxiv.org/abs/2004.05150) пошли вероятно по самому простому пути, решили ограничить множество токенов, которые участвуют в подсчете внимания. Благо для этого в Трансформере уже есть готовый механизм - маска внимания. Если это сделать, то сохранив общее кол-во активных токенов, доступный контекст можно серьезно расширить. Модель назвали Longformer = Long Document Transformer, чтобы как раз подчеркнуть основную цель данной модификации - увеличение доступного контекста. 

Авторы расммотрели несколько масок внимания. Во-первых, это скользящее окно из соседних токенов (локальное внимание). Во-вторых, окно предложили разреживать (dilated window), то есть используем каждый второй или каждый третий токен и т.п. Да, есть риск пропустить какой-то важный токен, но с большой вероятностью его конткст подскажет. В третьих, некоторые токены получают безусловное внимание, так как считаются важными, к таким относятся токены из самого начала контекста. 

Сложность становится линейной по n.

Иллюстрация масок ниже. В режиме авторегресионной генерации (GPT) дополнительно добавляется нижнетреугольная маска:

<img src="img/longformer.png" width=500>

### BigBird
[[Zaheer, 2021]](https://arxiv.org/abs/2007.14062) решили, что важно расширить маски внимания случайными токенами.

<img src="img/bigbird.png" width=500>

Теоретическое обоснование взято из теории графов: графами «Small World» (графами высокой связности) называют графы, где путь между любой парой вершин короткий. Добавление случайных рёбер в граф довольно быстро превращают малосвязный граф в сильно связванный (как бы добавляется возможность "телепортации" из вершины в вершину). Этот прицип является основой многих социальных эффектов, в частности теории шести рукопожатий.

На уровне одного слоя внимания какой-то особенной связности нет: запрос стандартно агрегирует сигнал от доступных ему токенов k. Но если мы рассматриваем модель целиком, как суперкпозицию множества слоев внимания, то видим, как сигнал может распространяться от токена к токену в том числе через добавленные случайные связи. 

Авторы показали, что такая схема приближает выразительность полного внимания, при этом оставаясь линейной по времени вычисления.



<img src="img/routing.png" width=450>

### Reformer
[[Kitaev et al, 2020]](https://arxiv.org/abs/2001.04451) из Google обратили внимание, что сам расчет внимания в Трансформерной архитектуре избыточен и предложили свою архитектуру, которую назвали Reformer (Reversable Transformer). В ней они реализовали три модификации.

Во-первых, они заметили что на практике внимание распределяется неравномерно, обычно оно достается всего нескольким токенам из контекста, а значение остальных оказывается около нуля. Соотвественно нас в первую очередь интересуют токены дающие максимальное произведение, а остальными можно пренебречь. Максимум скалряного произведения достигается, когда q и k сонаправлены, иными словами, чем ближе векторные представления k и q, тем больше связь между ними. Если сегментировать вектора и распредлеить их в однородные "корзины", тогда перебор можно ограничить рамками одной корзины.

Для этого они прибегли к идеально подходящему под эту задачу алгоритму приближенного поиска соседей LSH (locality-sensitive hashing). Идея в том, что все ветктора описываются небольшим набором случайными проекцияй, которые можно очень быстро посчитать и не тратить время на полный расчет расстояния между векторами. Тогда с большой вероятностью близкие или идентичные представления попадут в одним проекций. Останется перебрать пары из одной корзины.

<img src="img/reformer.png" width=600>

Во-вторых, обучение сети методом обратного распролстранения ошибки требует хранения активаций всех слоев, поэтому стандартно на прямом проходе оптимизатор их запоминает в памяти и на обратном проходе достает. Но если хочется сэкономить на памяти, на обратном проходе активации можно не запомнианть, а вычилсять повторно - это занимает больше времени, но меньше расходуется память. В случае с классическим Трансофрмером это невозможно, так как . Поэтому авторы предложили переконфигурировать остаточные связи - . Теперь тут нет рекурсии и взоды можно восстановить. Назали это обратимыми слоями (reversible residuals).

Третья новация связана с вычислением полносвязного слоя (FFN). Это вычисление использует самый большой тензор, что дает пиковое использование памяти. Но поскольку применяется независимо ко всем токенам входной последовательности, вычисление можно разбить на последовательные блоки. Общее кол-во не меняется, но пиковое испольование памяти уменьшается.

Статья дала старт целому направлению "Efficient Transformers", но мейнстримом модель так и не стала, уступив пальму первенства методам типа FlashAttention (о нем ниже).

### Routing Transformer
[[Roy et al, 2020]](https://arxiv.org/abs/2003.05997) вышли с похожей на Reformer идеей - сделать маску разреженности динамической, зависящей от входа, но вместо LSH они использовали кластеризацию. Модель назвали __Routing Transformer__, показывая, что. В качестве метода кластеризации использовали k-means (онлайн его вариант: примеры показываются один за другим, кластеризация уточняется). Каждый запрос $q$ считает внимание только с ключами $k$ своего же кластера и таким образом сокращается общее кол-во вычислений.

### Linformer
Для экономии вычислений [[Wang et al, 2021]](https://arxiv.org/abs/2006.04768) идут чуть по другому пути, они не разделяют последовательность токенов на сегменты, а плотно упаковывают всю эту последовательность представлений в небольшой фиксированной размерности вектор псевдотокенов. Делается это для представлений K и V через два добавленных перед вниманием слоя проекций, после чего вычисляется стандартный механизм внимания. В батчевом режиме он уже не квадратный, а прямоугольный $QK_{dense}^T$, то есть каждый оригинальный токен $q$ агрегирует сигнал от сжатого набора псевдотокенов $k$.

Модель назвали __Linformer__ акцентируя внимание на том, что из-за фикисрованной размерности последовательности псевдотокенов, сложность вычисления становится линейной: ведь независимо от размера входного контекста, нам всегда достаточно сравниться с k псевдотокенами.

<img src="img/linformer1.png" width=500>

### Performer
Команда [[Choromankspiy et al, 2022]](https://arxiv.org/abs/2009.14794) из Google также пошли по пути приближения, но более радикально, они предложили заменить саму формулу расчета внимания на ее приближенный вариант, основываясь на паре математически приемов, главным из которых было разложении ядра (kernel trick). Модель назвали __Performer__, а алгоритм приближенного внимания FAVOR+ (Fast Attention Via positive Orthogonal Random features). В результате упрощения формулы появилвась ассоциативность и умножать на V раньше, чем умножатьна Q. Тем самым вообще не материализовать квадратичную матрицу $QK^T$.

Напомним, что обычное внимание вычисляется как $\mathrm{Att}(Q,K,V)=D^{-1}AV$, где $A_{ij}=\exp(q_i^\top k_j/\sqrt d)$ и $D=\mathrm{diag}(A\mathbf 1_L)$. 

Построение матрицы $A$ размера $L\times L$, имеет квадратичную по длине контекста $L$ сложность. Авторы $A_{ij}=\mathcal K(q_i,k_j)$ предлагают представление $\mathcal K(q,k)=\mathbb E_{\omega}[\varphi_\omega(q)^\top\varphi_\omega(k)]$ через отображение в признаковое пространство. Если построить явное приближение $\varphi:\mathbb R^d\to\mathbb R^m$, то $A\approx\varphi(Q)\varphi(K)^\top$, и матрица факторизуется на два «узких» множителя.

Основная сложность в том, что не очень понятно, как выбирать конкретное отображение $\varphi$ в пространство фичей. По классике для разложения RBF ядра (экспоненты) используют тригонометрические случайные признаки:

$$\varphi_{\mathrm{trig}}(x) = \frac{1}{\sqrt{m}}\Big[\cos(\omega_1^\top x),\dots,\cos(\omega_m^\top x),\; \sin(\omega_1^\top x),\dots,\sin(\omega_m^\top x)\Big], \qquad \omega_i \stackrel{\text{iid}}{\sim} \mathcal{N}(0,I_d)$$

Но проблема с ними в том, что они дают знакопеременные координаты, из-за чего сложно приближать малые значения ядра — а именно такие значения доминируют в реальных матрицах внимания. Поэтому авторы предложили использовать другое представление со строго положительными признаками:

$$\varphi(x)=\frac{\exp(-\|x\|^2/2)}{\sqrt m}\Big[\exp(\omega_1^\top x),\dots,\exp(\omega_m^\top x)\Big],\qquad \omega_i\sim\mathcal N(0,I_d),$$

для которого $\mathbb E[\varphi(q)^\top\varphi(k)]=\exp(q^\top k)$ точно, а дисперсия стремится к нулю там же, где стремится к нулю само ядро. 

Дополнительно векторы $\omega_i$ генерируются попарно ортогональными, что строго уменьшает дисперсию оценки.

После замены $QK^T$ на приближенное представление становится возможной перестановка скобок в произведении, где вместо $(\varphi(Q)\varphi(K)^\top)V$ мы можем посчитать $\varphi(Q)\,(\varphi(K)^\top V)$. Знаменатель находится аналогично: $\hat D=\mathrm{diag}\big(\varphi(Q)(\varphi(K)^\top\mathbf 1_L)\big)$. То есть не материализовать проблемную квадратную матрицу, в просто считать проивзедение двух векторов. В авторегрессионном режиме внимание вообще обновляется за константное время $O(1)$, а длина последовательности нигде не зашита в веса, что делает подход гибким.

$$\mathrm{out}_i=\hat d_i^{-1}\,\varphi(q_i)^\top S_i,\qquad S_i=S_{i-1}+\varphi(k_i)v_i^\top\in\mathbb R^{m\times d},$$

На практике замена не очень окупилась: Performer уступил полному вниманию, поскольку приемлемая дисперсия требует довольно большой размерности $m$. Тем не менее важный пример того, что линейное внимание может быть выведено из аппроксимационных соображений. Похожий прием испольщовался в более поздней модели MAMBA (подробнее в следующей главе).

## Экстраполяция конекста
### Интерпоялция позиций
### NTK-scaling
### YARN

## Экономия KV-кэша
Второе слабое место Траснформерной архитектуры - это инференс, потокенная генерация ответа. Мы кэшируем ключи и значения всех прошлых токенов. Размер кэша пропорционален числу слоёв, числу голов, размерности головы, длине контекста и батчу — и именно он определяет, сколько памяти и пропускной способности съест декодирование.

### MQA
[[Shazeer, 2019]](https://arxiv.org/abs/1911.02150) предложили сыграть на избыточности внимания и оставить только по одному экзмепляру проекций для K и V векторов. Разнообращие обьеспечивается только для запросов Q, поскольку считает, наиболее важный компонент сигнала, для Q оригинальное количество представлений. Таким образом общее количество вычислений не меняется, но использование памяти под KV кэш заметно сокращается, а значит и генерация заметно ускоряется. Подход назвали __Multi-Query Attention__. За оптимизацию приходится платить некоторой просадкой качества и меньшей стабильностью обучения, поскольку ограничивается разнообразие представлений K и V.

<img src="img/mqa.webp" width=500>

### GQA
[[Ainslie et al, 2023]](https://arxiv.org/abs/2305.13245) предложили __Grouped-Query Attention__ компромисс между полным вниманием (MHA) и MQA. Можество запросов делятся на G групп, и каждая группа разделяет одну голову K/V. существующую MHA-модель можно дёшево «дообучить» (uptrain) в GQA. В том числе по этой причине GQA стал стандартом в моделях LLaMA-2/3, Mistral и других.

<img src="img/gqa.png" width=500>

### MLA
[[Liu et al, 2024]](https://arxiv.org/abs/2405.04434) __Multi-head Latent Attention__ (DeepSeek-V2) меняет сам вопрос. Вместо «как поделить меньшее число голов K/V» спрашивается «зачем вообще хранить полноразмерные K и V». K и V совместно сжимаются низкоранговой проекцией в маленький латентный вектор, и в кэше лежит только он; при вычислении внимания пер-головые K и V восстанавливаются обратной проекцией на лету. В DeepSeek-V2 это дало сокращение KV-кэша примерно на 93% относительно MHA, причём, по их измерениям, не ценой качества, а с небольшим выигрышем. 

Платой стала сложность: низкоранговое сжатие плохо дружит с RoPE, поэтому введён «расщеплённый» (decoupled) RoPE — отдельная небольшая часть размерностей несёт позиционную информацию; и приём «поглощения весов» (weight absorption), сворачивающий проекции, чтобы восстановление не стоило лишних вычислений. MLA архитектурно более инвазивен, чем GQA, но и потенциально мощнее: он торгует дополнительными вычислениями (распаковка) за резкое снижение памяти и трафика

<img src="img/mla.png" width=500>

### NSA
Команда [(Yuan et al, 2025)](https://arxiv.org/abs/2502.11089) из DeepSeek вернулись к старой идее разреженности со своим вариантом внимания __NSA (Native Sparse Attention)__. Они сформулировали пайплайн, состоящий из трех парадлельных экстракторов. Во-первых, это сжатый поиск: блоки токенов суммаризируются в компактные представлений. Во-вторых, точный поиск: выбираются самые релевантные блоки токенов, к которым затем применяется полное внимание. В третьих используется скользящее окно (свежий локальный контекст). Выходы всех трех компонентов смешиваются обучаемым гейтом. 

Два принципиальных отличия от методов 2020 года. Во-первых, NSA обучаема нативно — разреженность присутствует с самого предобучения, а не плявляется в первый раз на инференсе. Во-вторых, она аппаратно-согласована: шаблон спроектирован под GPU (сбалансированная арифметическая интенсивность, блочные ядра, выровненные под группировку GQA), поэтому теоретическая экономия операций превращается в реальное ускорение по времени. На последовательностях в 64k NSA заметно быстрее полного внимания на декодировании, прямом и обратном проходах, при этом не уступая ему в качестве.

<img src="img/nsa.png" width=500>

## FlashAttention
(Dao et al, 2022) задались вопросом, можно ли перекомпоновать алгоритм расчета, оставив его точным, но ускорить вычисление. Оказалось, что не только можно, но это еще и очень эффективно. Так родился алгоритм __FlashAttention__ модификация обычного самовнимания, ускоряющая расчет. Ключевое наблюдение — стандартное внимание упирается не в вычисления, а в память: оно записывает в медленную HBM огромную промежуточную матрицу n×n и читает её обратно. Значит, надо минимизировать обращения к HBM, а не число операций.

### FlashAttention-1
[[Dao et al, 2022]](https://arxiv.org/abs/2205.14135)<Br>
FlashAttention компонует вычисление внимания таким образом, что становится более оптимальным по I/O. На GPU есть быстрая SRAM память и медленная HBM, хочется больше вычислений делать на SRAM, 

Attention матрица нарезается на куски (процесс называется тайлинг): блоки Q, K, V подгружаются из HBM в быструю память SRAM и обрабатываются по частям. 
Softmax считается «онлайн» по частям (с бегущими максимумом и суммой), поэтому полная матрица внимания нигде не материализуется целиком. 
На обратном проходе используется пересчёт: вместо хранения большой матрицы её восстанавливают из компактной статистики. 

В итоге память линейна по n, число обращений к HBM резко падает, а итоговое ускорение по времени — в 2–4 раза, почти бесплатно и с сохранением точности.

<img src="img/flash1.png" width=600>

### FlashAttention-2
[[Dao et al, 2023]](https://arxiv.org/abs/2307.08691)<br>
__FlashAttention-2__, вторая версия (2023), не меняет алгоритм, но переписывает распараллеливание. Сокращается доля «не-matmul» операций (тензорные ядра GPU считают матричное умножение во много раз быстрее, чем специальный блок, отвечающий за экспоненту в softmax), вычисление параллелится вдоль длины последовательности, а работа лучше распределяется между варпами и блоками, уменьшая трафик через разделяемую память. Это даёт ещё около двукратного ускорения и доводит утилизацию до примерно 50–70% на A100. На H100, однако, версия достигала лишь ~35%, потому что не использовала особенности нового железа

### FlashAttention-3
В 2024 году [[Shah et al, 2024]](https://arxiv.org/abs/2407.08608) выпустили __FlashAttention-3__, третью версия алгоритма. В этот раз целью была максимально возможная синхронизацию с железом, а конкренто с архитектурой Nvidia Hopper (H100). Авторы при этом не отходят от своего главного принципа - внимание по-прежнему остается точным.

Три приёма: использование асинхронности (warp-specialization — одни варпы через TMA асинхронно подгружают данные, другие в это время считают на тензорных ядрах WGMMA, перекрывая память и вычисления); чередование (ping-pong) блочного matmul и softmax, чтобы медленная экспонента считалась одновременно с матричным умножением; и низкая точность FP8 с блочным квантованием и «incoherent processing», которые удерживают точность (примерно в 2,6 раза меньше ошибка, чем у наивного FP8). 

В результате простой на видеокартах H100 сокращается с 65% до 25%, а скорость генерации становится в 1.5–2 раза выше второй версии.

Алгоритм FlashAttention совершил своего рода революцию в вычислениях языковых моделей на видеокартах и фактически обесценил целое направление приближённых методов,  и показав, что того же эффекта можно добиться с сохранением точности и сделав их бесполезными.

## Спекулятивное декодирование
[(Leviaithan et al, 2023)](https://arxiv.org/pdf/2211.17192) описали идею спекулятивного декодирования (speculative decoding), основой которой является наблюдение, что генерация обычно неравномерна по сложности. Какие-то куски ответа сгенерировать просто (```2 x 2 = ```), для каких-то нужно серьезное рассуждение. Почему бы не переключаться между разными моделями на ходу? Допустим есть большая языковая модель (целевая, target), и есть компактная языковая модель (черновая, draft). Дадим малой модели возможность быстро генерировать продолжение на несколько шагов, а затем большая модель проверит качество ее продолжения. Если оно удовлетворительное, то идем дальше, если неудовлетворительное, целевая модель перегенерирует то же самое продолжение, но уже самостоятельно.

Шаг верификации продолжения дешевый, поскольку происходит за одну итерацию Траснформера может проверяться сразу много нагенерированных черновых токенов. В роли модели-черновика часто выбирают более базовую версия той же целевой модели. Приемка вероятностная: целевая модель сравнивает две вероятности сегенерированного продолжения, свою $P(t)$ и дочерней модели $Q(t)$. Если дочерняя модель переоценила вероятность токена $(Q > P)$, продолжение принимается с вероятностью $P/Q$. Если недооценила, заменяем на вариант целевой модели.

Таким образом, выигрыш есть, если черновик дёшев, а доля принятых токенов высока.

### Blockwise Parallel Decoding
В 2018 году еще до появления самого термина "спеулятивное декодирование" описали модель Blockwise Parallel Decoding. Идея простая - к выходу последнего скрытого состояния модели добавляется несколько лёгких «голов» (heads), каждая из которых независимо предсказывает токен на позиции +1, +2, +3 и так далее.

### Medusa
В области генеративных языковых моделей существует целый набор методов, озаглавленный неавторегрессионная генерация (NAT), основная идея которого - генерация продолжения не из одного, а сразу из сразу нескольких токенов. 
Первая модель была вообще без верификации . Позже появилась модель Blockwise Parallel Decoding. 

[[Cai et al, 2024]](https://arxiv.org/abs/2401.10774) взяли старую модель BPD и предложили пару модификаций процесса верификации. Главная доработка - вместо генерации одного продолжения генерируется сразу множество и организуется в виде дерева. Так появилась модель __Medusa__.  В первой версии модели Medusa-1 достаточно обучить только головы, остальные веса замораживается, что обсепечивает высокую скорость работы. Во второй версии модель обучается целиком.

Как генерируется дерево продолжений? Каждая голова генерирует top-k токенов и по этим наборам строится декартово произведение всех возможных комбинаций, на основании которой строится дерево. Далее все комбинации укладываются в одну плоскую структуру - это сделано, чтобы можно было посчитать вероятности каждого токена за одну итерацию. А чтобы гарантировать, что токен видит только свой префикс используется маска внимания. В MEDUSA такую маску внимания называют Tree Attention. И затем остается верифицировать все сгенерированные продолжения, выбрав наиболее вероятное.

<img src="img/medusa1.png" width=300>

В работе описано три механизма верификации. В рамках "жадной" стратегии, мы просто на каждом шаге выбираем токен с наибольшей вероятностью по целевой модели. В рамках стратегии "Typical Acceptance" мы считаем вероятность каждого токена по основной модели и сравниваем ее с порогом уверенности, скорректированным на энтропию его распределния (при малой энтропии порог уменьшается). В рамках стратегии `nucleus` для каждой позиции в дереве алгоритм берёт распределение вероятностей всех токенов, полученное от основной модели, сортирует их по убыванию и начинает добавлять токены в так называемое «ядро», пока суммарная вероятность накопленных токенов не достигнет порога `top_p` (обычно 0.8). Токен-кандидат от головы MEDUSA принимается, если он попал в это ядро, и отклоняется в противном случае.

Явный минус подхода MEDUSA в том, что каждая голова генерирует токен независимо от других (находятся вне локального контекста, особенно последние токены), поэтому точность генерации явно падает.

### Hydra
[(Ankner, 2024)](https://arxiv.org/abs/2402.05109) решили исправить эту независимость и сделали генерирующие головы (heads) последовательно зависимыми: каждая получает на вход токены, предложенные предыдущими. По сути черновик превращается из набора независимых предсказанных токенов в нормальную последовательную модель, что заметно повышает среднюю длину принятого фрагмента.

<img src="img/hydra.png" width=300>

### EAGLE
[[Li et al, 2024]](https://arxiv.org/abs/2401.15077) ключевая мысль первой версии модели __EAGLE-1__ (Extrapolation Algorithm for Greater Language-model Efficiency): генерировать следующий токен не по токеном, но и скртые преставение  предпоследнего скрытого состояния. Идея в том, что непрерывное скрытое представление гораздо лучше в качестве сигнала, чем дискретные токены. затем из предсказанного признака получают токен через готовую LM-голову целевой модели. Скрытое состояние передается вместе со сгенерированным токеном.

Черновик делает фиксированное число авторегрессионных шагов, обычно 4-8. Затем верификатор (полнаая модель) оценивает и оставляет наиболее длинную принятую последовательность. Черновая модель представляет сильно усеченную версию основной модели: слой эмбединов и выходной HEAD слой берутся из оригинальной модели, а между ними добавляется один траснформеный слой. Модель в авторегрессионном режиме генерирует продолжение и . Для верификации заимствуется идея из MEDUSA, генерируется сразу дерево возможных продолжений, которые оцениваются механикой TreeAttetnion.

<img src="img/eagle1_1.png" width=500>

[(Li et al, 2024)](https://arxiv.org/abs/2406.16858) во второй версии модели __EAGLE-2__ корректировку решили делать не после генерации чернового дерева, а в процессе его построения с помощью beam search. Черновая модель сама оценивает уверенность в ответе.

[[Li et al, 2025]](https://arxiv.org/abs/2503.01840) в версии __EAGLE-3__ решили отказаться от предсказания скрытого состояния, а предсказывать сразу токен. Вместо одних только верхних признаков используется конкатенация признаков с разных слоев. Кроме того, многошаговый процесс черновика симулируется уже на обучении, устраняя рассинхрон между обучением и инференсом. Это даёт порядка 3–6,5× ускорения относительно обычной генерации и на 20–40% выше EAGLE-2.


С чем они себя сравнивают:
- Transformer-XL (2019)
- Adaptive Span (2019)
- Compressive (2020)
- Reformer (2020)
- Sparse (2019)
- Routing (2020)
- BP-Transformer (2019)
- Blockwise (2019)

# Спекулятивное декодирование: методы за пределами Medusa, Hydra и EAGLE

Этот материал дополняет главу, в которой уже разобраны базовый алгоритм спекулятивного декодирования (Leviathan et al., 2023; Chen et al., 2023), а также Medusa, Hydra и EAGLE. Здесь собраны остальные значимые методы. Для каждого сформулированы две вещи: в чём состоит его собственная идея и чем он отличается от того, что было до него.

---

## 0. Рамка: три оси, по которым различаются все методы

Прежде чем перечислять методы, полезно задать читателю систему координат. Почти каждая работа в этой области отвечает на три независимых вопроса, и различия между методами удобнее объяснять как разные комбинации ответов.

**Первая ось — откуда берётся черновик.** Черновик может генерировать отдельная маленькая модель (классическая схема), сама целевая модель в облегчённом режиме (self-drafting), дополнительные головы поверх целевой модели (Medusa, EAGLE, MTP) или вообще не модель, а поиск по тексту (REST, LLMA, Prompt Lookup).

**Вторая ось — какова структура черновика и как устроена верификация.** Черновик может быть линейной цепочкой токенов, деревом кандидатов, проверяемым за один проход с помощью tree attention, или решёткой n-грамм, порождённой итерацией Якоби. Отдельный вопрос — какой статистический критерий приёма используется и сохраняет ли он распределение целевой модели.

**Третья ось — lossless или lossy.** Строгое speculative sampling гарантирует, что выход побитово соответствует выборке из целевой модели. Ряд методов сознательно отказывается от этой гарантии ради более высокой доли принятых токенов.

Держа эти три оси в голове, читатель перестаёт воспринимать методы как разрозненный список и начинает видеть в них комбинации.

---

## 1. Предыстория: что было до Leviathan и Chen

### Blockwise Parallel Decoding (Stern et al., 2018)

**Идея.** Авторы добавляют к трансформеру `k` дополнительных выходных голов, каждая из которых предсказывает не следующий токен, а токен на позиции `+2`, `+3`, ..., `+k`. За один проход модель выдаёт блок из `k` кандидатов, после чего тот же трансформер проверяет их одним параллельным прогоном и принимает самый длинный корректный префикс.

**Чем отличается.** Это не отличие, а исток: перед вами прямой предок Medusa, опубликованный за пять лет до неё. Разница в том, что Stern и соавторы работали в жадном режиме и не имели ни tree attention, ни статистически корректного правила приёма для сэмплирования. Показать читателю эту работу полезно именно затем, чтобы Medusa перестала выглядеть изобретением с нуля: Medusa — это блочные головы Стерна плюс дерево кандидатов плюс typical acceptance.

### SpecDec / Aggressive Decoding (Xia et al., 2022–2023)

**Идея.** Работа впервые вводит сам термин «speculative decoding». Для задач, где выход сильно перекрывается со входом (исправление грамматики, постредактирование), черновиком служит непосредственно входной текст: модель проверяет, совпадает ли её собственный выход с входом, и продолжает копировать, пока совпадение держится. В более общем варианте авторы формулируют принципы проектирования драфтера: он должен быть не «просто маленькой авторегрессионной моделью», а неавторегрессионным декодером с независимыми головами, глубоким энкодером и мелким декодером.

**Чем отличается.** В отличие от последующих работ, здесь ускорение достигается не за счёт обученного драфтера, а за счёт структурного свойства задачи. Это первый пример того, что позже разовьётся в целое семейство методов копирования (LLMA, Prompt Lookup).

---

## 2. Черновик без модели: копирование и поиск

Это семейство отвечает на первую ось радикально: драфтера нет вообще, кандидаты берутся из текста. Практическая ценность здесь максимальна, потому что такие методы не требуют ни обучения, ни дополнительных весов, ни чекпойнта под конкретную модель.

### LLMA / Inference with Reference (Yang et al., Microsoft, 2023)

**Идея.** Во многих сценариях выход модели содержит длинные дословные фрагменты из контекста: в retrieval-augmented генерации модель цитирует найденные документы, в многоходовом диалоге повторяет формулировки предыдущего хода, при редактировании кода переписывает файл с точечными правками. Метод отслеживает совпадение суффикса уже сгенерированного текста с фрагментами reference-текста и, найдя совпадение, копирует продолжение оттуда как черновик.

**Чем отличается.** По сравнению с классической схемой Leviathan/Chen здесь полностью устранена стоимость драфтинга: копирование n-граммы стоит микросекунды против полного прохода маленькой модели. Плата — крайне неравномерная эффективность: на задачах без перекрытия входа и выхода метод не даёт ничего.

### Prompt Lookup Decoding (Saxena, 2023)

**Идея.** Это предельно минималистичная версия предыдущего метода: суффикс генерации длиной 2–3 токена ищется в промпте обычным сопоставлением строк, и найденное продолжение становится черновиком.

**Чем отличается.** От LLMA метод отличается тем, что отказывается даже от отдельного reference-хранилища и работает только с промптом, а реализация занимает несколько десятков строк. Педагогически он бесценен: на нём проще всего показать, что «драфтер» в спекулятивном декодировании — это произвольный дешёвый источник гипотез, а не обязательно нейросеть. В vLLM этот метод присутствует под именем `ngram`.

### REST: Retrieval-Based Speculative Decoding (He et al., 2023)

**Идея.** Метод заменяет промпт как источник кандидатов на внешнее хранилище: по большому корпусу строится суффиксный массив, суффикс текущей генерации используется как запрос, найденные продолжения агрегируются в префиксное дерево (Trie), из которого отбираются наиболее частотные ветви, и полученное дерево кандидатов верифицируется целевой моделью за один проход.

**Чем отличается.** От LLMA и Prompt Lookup метод отличается тем, что источник кандидатов не ограничен текущим контекстом и потому работает на задачах без перекрытия входа и выхода. От классического драфтера он отличается тем, что «обучение» сводится к построению индекса, а не к градиентному спуску: добавить новый домен — значит дописать данные в хранилище. Ключевое ограничение, которое стоит подчеркнуть: качество черновика напрямую определяется тем, насколько датастор соответствует домену запросов.

### Suffix Decoding (Oliaro et al., Snowflake, 2025)

**Идея.** Метод строит суффиксное дерево не по статическому корпусу, а по истории предыдущих выходов самой системы и по текущему промпту, и подбирает длину и форму черновика адаптивно, исходя из статистики этого дерева.

**Чем отличается.** От REST метод отличается динамичностью хранилища: оно наполняется на лету и потому автоматически подстраивается под нагрузку. Это принципиально важно для агентных сценариев, самокоррекции и повторяющихся вызовов инструментов, где модель раз за разом порождает почти одинаковые фрагменты. В vLLM метод доступен как `suffix`.

### Token Recycling (2024)

**Идея.** На каждом шаге модель порождает распределение по всему словарю, но обычная схема использует из него один токен, а остальные top-k кандидаты выбрасываются. Метод сохраняет их в матрице смежности «токен → его вероятные продолжения» и на следующих шагах собирает из этой матрицы дерево черновика.

**Чем отличается.** От всех предыдущих методов он отличается источником сигнала: не текст и не отдельная модель, а побочный продукт вычислений самой целевой модели, который до этого просто терялся. Метод не требует ни обучения, ни хранилища, ни дополнительной памяти сверх небольшой матрицы.

### SAM-Decoding (2025)

**Идея.** Вместо суффиксного дерева используется суффиксный автомат, что даёт амортизированное O(1) на шаг вместо логарифмического поиска, а сам метод комбинирует статический датастор (как в REST) с динамическим (как в Suffix Decoding).

**Чем отличается.** Это инженерное развитие линии REST/Suffix Decoding: идея та же, выигрыш в асимптотике поиска, за счёт чего накладные расходы драфтинга становятся пренебрежимыми даже при высокой частоте шагов.

### Ouroboros (2024)

**Идея.** Метод поддерживает пул фразовых кандидатов, который пополняется за счёт токенов, отвергнутых на предыдущих шагах верификации, и использует этот пул для удлинения черновика, порождённого обычной draft-моделью.

**Чем отличается.** В отличие от чисто поисковых методов, Ouroboros не заменяет драфтера, а надстраивается над ним: отвергнутые токены перестают быть чистой потерей и превращаются в материал для будущих черновиков.

---

## 3. Модель драфтит сама себя

Это семейство отвечает на первую ось иначе: отдельной draft-модели нет, но и дополнительных голов, как в Medusa, тоже нет. Черновик порождает сама целевая модель в удешевлённом режиме.

### Self-Speculative Decoding / Draft & Verify (Zhang et al., 2023)

**Идея.** Черновик генерирует сама целевая модель с пропуском части промежуточных слоёв. Какое подмножество слоёв пропускать, подбирается офлайн байесовской оптимизацией на небольшой выборке. Дополнительно вводится адаптивный критерий выхода из драфтинга: если уверенность падает, генерация черновика прекращается досрочно, не дожидаясь фиксированной длины.

**Чем отличается.** От классической схемы метод отличается тем, что не требует ни отдельной модели, ни дополнительных весов, ни обучения — а значит, снимает главную практическую проблему всего направления, необходимость иметь совместимый драфтер под каждую целевую модель. От Medusa и EAGLE он отличается тем, что не требует обучения голов. Плата — более скромное ускорение, порядка 1.3–1.8×, поскольку пропуск слоёв удешевляет проход не так сильно, как маленькая модель.

### LayerSkip (Meta, 2024)

**Идея.** Модель заранее обучается с layer dropout и с дополнительным лоссом раннего выхода, чтобы её промежуточные слои давали осмысленные предсказания. При инференсе черновик получают ранним выходом из слоя `l`, а верификацию проводят, достраивая оставшиеся слои поверх уже вычисленных активаций.

**Чем отличается.** От Draft & Verify метод отличается тем, что переносит работу на этап обучения и потому получает существенно более качественные ранние выходы. Ещё важнее системное отличие: черновик и верификация делят один KV-кэш, поэтому верификация не требует пересчёта нижних слоёв и обходится почти бесплатно. Это одна из немногих схем, где стоимость проверки действительно близка к нулю.

### Kangaroo (2024)

**Идея.** В качестве драфтера используется фиксированная нижняя часть целевой модели плюс обученный лёгкий адаптер, который «доводит» промежуточные представления до пригодных для предсказания токена. Дополнительно применяется двойной ранний выход: не только по слоям, но и по шагам драфтинга — при низкой уверенности драфтинг останавливается.

**Чем отличается.** От LayerSkip метод отличается тем, что не требует переобучения целевой модели: обучается только адаптер. От Medusa и EAGLE он отличается тем, что драфтер остаётся авторегрессионным и переиспользует настоящие слои модели, а не предсказывает несколько позиций независимыми головами.

### SWIFT (2024)

**Идея.** Набор пропускаемых слоёв определяется прямо во время инференса, по ходу обработки конкретного запроса, а не подбирается заранее.

**Чем отличается.** От Draft & Verify метод отличается отсутствием офлайн-стадии оптимизации, что делает его полностью plug-and-play, и адаптивностью к домену: оптимальное подмножество слоёв для кода и для диалога различается.

### Speculative Streaming (Apple, 2024)

**Идея.** Вместо отдельных голов Medusa метод вводит multi-stream attention: внутри одной и той же модели поддерживается несколько «потоков», каждый из которых отвечает за свою будущую позицию, и они обмениваются информацией через механизм внимания.

**Чем отличается.** От Medusa метод отличается тем, что головы не независимы (это сближает его с Hydra), и тем, что число дополнительных параметров на порядки меньше — что критично для развёртывания на устройствах.

### PaSS: Parallel Speculative Sampling (2023)

**Идея.** К входу добавляются специальные обучаемые look-ahead эмбеддинги, играющие роль «пустых мест» под будущие токены; за один проход модель заполняет их, порождая черновик.

**Чем отличается.** От Medusa метод отличается тем, что не добавляет выходных голов и обучает лишь несколько эмбеддингов, то есть требует минимального дообучения. Ускорение при этом скромнее.

### Lookahead Decoding (Fu et al., 2024)

**Идея.** Метод параллельно выполняет итерации Якоби: он поддерживает «окно» из нескольких будущих позиций и на каждом шаге одновременно уточняет их все, а также собирает n-граммы, возникшие в ходе этих итераций, в пул. Пул служит источником кандидатов, которые верифицируются вместе с обычным шагом декодирования.

**Чем отличается.** Это единственный из перечисленных методов, в котором нет ни драфтера, ни голов, ни поиска по тексту: черновик порождается самим процессом параллельного решения системы уравнений авторегрессии. Отличие от всех предыдущих — характер платы за ускорение. Другие методы экономят FLOPs, Lookahead их сознательно тратит: он загружает простаивающие вычислительные блоки GPU, обменивая FLOPs на латентность. Отсюда следует важное практическое ограничение: метод хорош при малом батче и бесполезен при большом, когда GPU уже загружен.

### CLLM: Consistency LLM (2024)

**Идея.** Модель дообучают так, чтобы из произвольной начальной точки она за одну итерацию Якоби сходилась к неподвижной точке — то есть сразу выдавала правильное продолжение всего окна.

**Чем отличается.** От Lookahead Decoding метод отличается тем, что переносит стоимость с инференса на обучение, по аналогии с тем, как consistency-модели относятся к диффузионным. Число итераций Якоби сокращается радикально, но требуется дообучение целевой модели.

### Multi-Token Prediction (Gloeckle et al., 2024; MTP-головы DeepSeek-V3 и последующих моделей)

**Идея.** Модель предобучается с несколькими выходными головами, предсказывающими токены на несколько позиций вперёд. Изначально это делалось ради качества (такая задача оказывается лучшим обучающим сигналом), но побочным эффектом становится готовый драфтер: на инференсе те же головы порождают черновик.

**Чем отличается.** От Medusa метод отличается моментом обучения голов: не дообучение поверх замороженной модели на ограниченных данных, а полноценное совместное предобучение. Отсюда заметно более высокая доля принятых токенов. Практическое следствие, которое стоит подчеркнуть в учебнике: если модель изначально обучалась с MTP, отдельный чекпойнт драфтера не нужен вовсе, и именно поэтому MTP наряду с EAGLE-3 стал стандартом развёртывания к 2026 году.

---

## 4. Отдельная draft-модель: выравнивание распределений и каскады

Это семейство сохраняет классическую схему с отдельным драфтером, но задаётся вопросом: как сделать драфтера более согласованным с таргетом и как удешевить сам драфтинг.

### DistillSpec (Zhou et al., ICLR 2024)

**Идея.** Драфтер дистиллируется под конкретную целевую модель. Работа систематически исследует два фактора: какие данные использовать (оказывается, важны on-policy данные, то есть собственные генерации драфтера, а не фиксированный корпус) и какую дивергенцию минимизировать (оптимальный выбор зависит от режима декодирования — для жадного и для сэмплирования выигрывают разные дивергенции).

**Чем отличается.** От базовой схемы, где драфтером служит готовая маленькая модель того же семейства, метод отличается тем, что превращает выбор драфтера в задачу обучения с явной целью — максимизировать долю принятых токенов, а не качество драфтера самого по себе. Это принципиальный сдвиг: хороший драфтер здесь не тот, что хорошо предсказывает текст, а тот, что хорошо предсказывает *целевую модель*. Метод также предлагает lossy-вариант с ослабленным критерием приёма.

### Online Speculative Decoding (Liu et al., 2023)

**Идея.** Драфтер дообучается прямо в продакшене на распределении реальных запросов. Материал для обучения возникает бесплатно: каждый отвергнутый токен — это пример, где целевая модель уже выдала правильный ответ, то есть готовая пара «вход — метка от учителя».

**Чем отличается.** От DistillSpec метод отличается тем, что дистилляция становится непрерывной и адаптируется к сдвигу распределения запросов. Он опирается на наблюдение, что в реальном сервисе запросы кластеризуются по доменам, и узкоспециализированный драфтер бьёт универсальный.

### Staged Speculative Decoding (Spector & Ré, 2023)

**Идея.** Работа вносит два изменения одновременно: черновик реструктурируется из линейной последовательности в дерево, и, что важнее, сам драфтер ускоряется второй ступенью спекуляции — у него появляется собственный, ещё более дешёвый драфтер.

**Чем отличается.** От базового алгоритма метод отличается признанием того, что стоимость драфтинга не пренебрежима и её саму нужно оптимизировать теми же средствами. Отсюда идея рекурсии, которую следующая работа доводит до общего вида.

### Cascade Speculative Drafting / CS Drafting (Chen et al., 2023)

**Идея.** Метод вводит два ортогональных каскада. *Вертикальный каскад* рекурсивно применяет спекулятивное декодирование к самому драфтеру: большую модель драфтит средняя, среднюю — маленькая, и так вплоть до статистической n-граммной модели на дне иерархии, которая вообще не требует прохода по нейросети. *Горизонтальный каскад* распределяет вычислительный бюджет по позициям внутри черновика: ранние позиции получают более качественного (и дорогого) драфтера, поздние — более дешёвого.

**Чем отличается.** Вертикальный каскад обобщает идею Staged Speculative Decoding на произвольную глубину. Горизонтальный каскад — оригинальный вклад, и его стоит подчеркнуть отдельно, потому что он опирается на важное количественное наблюдение: вероятность принятия `i`-го токена черновика падает экспоненциально с ростом `i`, а значит, тратить на дальние позиции столько же вычислений, сколько на ближние, экономически неоправданно. Это же наблюдение объясняет читателю, почему нельзя просто увеличить длину черновика и получить произвольное ускорение.

---

## 5. Структура черновика и верификация

Эту тему стоит выделить в отдельный раздел: она ортогональна вопросу о происхождении черновика, и без неё непонятно, откуда в Medusa и EAGLE берутся деревья.

### SpecInfer (Miao et al., ASPLOS 2024)

**Идея.** Черновик порождается не одной моделью, а ансамблем нескольких маленьких моделей (дообученных техникой boost-tuning так, чтобы покрывать разные режимы), их гипотезы объединяются в дерево токенов, и всё дерево верифицируется целевой моделью за один проход с помощью специальной маски внимания — tree attention. Правило приёма обобщено с цепочки на дерево так, чтобы сохранить распределение целевой модели.

**Чем отличается.** От базового алгоритма метод отличается переходом от одной гипотезы к множеству: если один драфтер ошибся на позиции `i`, вся дальнейшая цепочка отбрасывается, тогда как в дереве шанс, что хотя бы одна ветвь верна, существенно выше. Практически важно донести до читателя, что **tree attention и верификация дерева — изобретение именно этой работы**, а Medusa, Hydra и EAGLE его унаследовали. Если в главе дерево впервые появилось в разделе про Medusa, это стоит явно оговорить.

### SpecTr (Sun et al., NeurIPS 2023)

**Идея.** Работа даёт теоретическую основу для приёма токенов при наличии нескольких черновиков, формулируя задачу как задачу оптимального транспорта с ограничением на мембранную структуру. Из этой постановки выводится алгоритм отбора, сохраняющий распределение таргета, и верхние границы на достижимое ускорение.

**Чем отличается.** SpecInfer предложил дерево как инженерное решение; SpecTr отвечает на вопрос, что вообще достижимо и какое правило приёма оптимально. Это единственная работа в списке, которую стоит давать читателю ради математики, а не ради практики.

### Block Verification (Sun et al., 2024)

**Идея.** Вместо последовательной проверки токен за токеном черновик верифицируется как единая последовательность, и авторы показывают, что такое блочное правило приёма оптимально в смысле ожидаемого числа принятых токенов.

**Чем отличается.** От классического правила Leviathan/Chen метод отличается тем, что оно оказывается субоптимальным: пошаговая проверка принимает решение жадно, не учитывая, что менее вероятный на текущем шаге токен мог бы открыть более длинное принятое продолжение. Выигрыш составляет порядка 5–8% при полном сохранении losslessness — то есть это буквально бесплатное улучшение.

### Sequoia (2024)

**Идея.** Структура дерева черновика не задаётся эвристически, а вычисляется динамическим программированием как оптимальная при заданных характеристиках оборудования (сколько токенов целевая модель успевает проверить за один проход без потери скорости). Дополнительно применяется сэмплирование без возвращения, что делает метод устойчивым к изменению температуры.

**Чем отличается.** От SpecInfer и Medusa, где форма дерева подбирается вручную и фиксируется, метод отличается тем, что превращает выбор дерева в задачу оптимизации с явной целевой функцией и явными аппаратными ограничениями. Отсюда же его устойчивость: фиксированные деревья заметно теряют в эффективности при высокой температуре, оптимизированные — нет.

### SpecExec (2024)

**Идея.** Метод строит очень большие деревья черновика — в сотни узлов — и рассчитан на сценарий оффлоадинга, когда веса целевой модели лежат в оперативной памяти или на диске, а не в памяти GPU.

**Чем отличается.** От всех предыдущих методов он отличается точкой в пространстве компромиссов. Когда один проход целевой модели стоит очень дорого (потому что требует перекачки весов через шину), рационально максимально нагрузить этот проход работой. То, что в обычном сценарии было бы расточительством, здесь оказывается оптимальной стратегией. Это хороший пример для учебника: он показывает, что «оптимальная длина черновика» — не константа, а функция от соотношения стоимостей.

### EAGLE-2 и OPT-Tree (2024)

**Идея.** Структура дерева определяется динамически, на лету, исходя из уверенности драфтера в текущем контексте: в позициях, где драфтер уверен, дерево вытягивается вглубь, где не уверен — ветвится вширь.

**Чем отличается.** От статических деревьев SpecInfer и Medusa эти методы отличаются тем, что используют информацию, доступную только во время инференса. Если в главе EAGLE описан в версии 1, дополнить его EAGLE-2 обязательно: динамическое дерево даёт больший прирост, чем многие «самостоятельные» методы из этого списка.

### Traversal Verification (2025)

**Идея.** Верификация дерева выполняется от листьев к корню и принимает решение на уровне целых последовательностей, а не отдельных токенов.

**Чем отличается.** Это перенос идеи Block Verification с цепочки на дерево: обычная верификация дерева сверху вниз отбрасывает ветвь, как только отвергнут её узел, тогда как посегментный критерий позволяет принять более длинное продолжение.

---

## 6. Отказ от гарантии эквивалентности: lossy-методы

Этот раздел стоит выделить явно, потому что базовый алгоритм подан как строго lossless, и читателю нужно понимать, что часть работ сознательно жертвует этим свойством.

### BiLD: Big Little Decoder (Kim et al., NeurIPS 2023)

**Идея.** Метод вводит две политики. *Fallback policy*: маленькая модель генерирует самостоятельно до тех пор, пока её уверенность превышает порог, и передаёт управление большой, когда уверенность падает. *Rollback policy*: большая модель проверяет сгенерированный фрагмент и откатывает его с точки, где расхождение её предсказания с предсказанием маленькой модели превысило порог.

**Чем отличается.** Ключевое отличие, которое обязательно нужно проговорить: **здесь нет rejection sampling и нет гарантии эквивалентности выходу целевой модели**. Выход BiLD — это выход гибридной системы, а не выборка из большой модели. Взамен большая модель вызывается существенно реже, чем в строгой схеме, где она проверяет каждый токен. Методически BiLD полезно противопоставить базовому алгоритму как «второй полюс» дизайн-пространства: спекулятивное декодирование Leviathan/Chen выбирает точность и получает ограниченное ускорение, BiLD выбирает ускорение и получает контролируемую, но реальную потерю качества.

### Typical acceptance (Medusa) и relaxed acceptance (DistillSpec)

**Идея.** Токен принимается не по строгому вероятностному критерию, а если его вероятность под целевой моделью просто «достаточно велика» — превышает порог, зависящий от энтропии распределения.

**Чем отличается.** Формально это тоже отказ от losslessness, но мягкий и хорошо контролируемый. Аргумент авторов состоит в том, что при сэмплировании с температурой мы и так не воспроизводим распределение модели точно, поэтому строгость критерия приёма — избыточная роскошь. Читателю стоит показать, что даже «lossless» Medusa в рекомендованной конфигурации lossless не является.

### Judge Decoding (2025)

**Идея.** Решение о приёме принимает обученный лёгкий классификатор, оценивающий не совпадение с целевой моделью, а приемлемость токена как такового.

**Чем отличается.** Метод исходит из наблюдения, что строгая верификация отвергает множество токенов, которые человек счёл бы полностью корректными — просто целевая модель выбрала бы синоним. Это доводит логику lossy-ветки до предела: критерием становится качество, а не тождество.

---

## 7. Системный слой: почему в продакшене цифры другие

Этот раздел я считаю самым важным дополнением к главе. Без него читатель уйдёт с завышенными ожиданиями, потому что практически все ускорения из статей измерены при размере батча, равном единице.

**Основной механизм.** Спекулятивное декодирование выигрывает потому, что авторегрессионная генерация упирается в пропускную способность памяти, а не в вычисления: GPU простаивает, перекачивая веса. Верификация нескольких токенов сразу использует этот простой. Но с ростом размера батча система становится вычислительно-ограниченной, простоя больше нет, и спекуляция превращается в чистые накладные расходы. В замерах на vLLM ускорение EAGLE падает примерно с 1.96× при батче 1 до примерно 1.21× при батче 128, поэтому в продакшене спекуляцию обычно автоматически отключают выше некоторого порога загрузки.

**SmartSpec и TurboSpec.** Идея состоит в том, чтобы выбирать длину спекуляции динамически, исходя из оценки goodput при текущей загрузке сервера, а не фиксировать её конфигом. Отличие от всех алгоритмических работ — целевая функция: оптимизируется не ускорение одного запроса, а пропускная способность системы при соблюдении SLO.

**PEARL (2024).** Метод устраняет взаимное ожидание драфтера и таргета, вводя предварительную верификацию первого токена черновика параллельно с драфтингом остальных. Отличие от классической схемы в том, что там драфтер и таргет работают строго по очереди, и один из них всегда простаивает.

**TriForce и MagicDec (2024).** При длинном контексте узким местом становятся не веса, а KV-кэш, поэтому черновик здесь строится на разреженном или усечённом KV самой целевой модели. MagicDec приносит контринтуитивный результат: при достаточно длинном контексте спекулятивное декодирование выгодно **и** при больших батчах, поскольку бутылочное горлышко сместилось.

**Spec-Bench (Xia et al., 2024).** Стандартный бенчмарк для сравнения методов на одном железе и одних данных. Упомянуть его стоит как методологическую опору: заявленные в разных статьях ускорения между собой несопоставимы, потому что различаются модели, длины, оборудование и режим декодирования.

---

## 8. Что появилось в 2025–2026

Эти работы свежие, и детали стоит перепроверить по первоисточникам.

**EAGLE-3.** Метод отказывается от предсказания фичи (что было центральной идеей EAGLE-1) в пользу прямого предсказания токена, обучает драфтер приёмом training-time test, имитирующим многошаговый инференс во время обучения, и подаёт на вход драфтеру смесь скрытых состояний нескольких слоёв, а не только последнего. Отличие от EAGLE-2 — снятие ограничения, которое мешало драфтеру улучшаться при росте объёма обучающих данных.

**EAGLE-3.1.** Совместная работа команд EAGLE, vLLM и TorchSpec, направленная на устойчивость: авторы диагностируют явление attention drift, при котором с ростом глубины спекуляции драфтер смещает внимание с sink-токенов на собственные сгенерированные токены, из-за чего качество деградирует на длинном контексте и нестандартных системных промптах.

**P-EAGLE.** Драфтер порождает все спекулятивные токены параллельно, а не последовательно, что снижает латентность драфтинга.

**Mirror Speculative Decoding (Apple, 2026).** Схема делается двусторонней: драфтер спекулирует продолжения для таргета, а таргет одновременно спекулирует корректирующие пути для драфтера. Отличие от всех предыдущих методов — отказ от асимметрии ролей.

**Speculative Speculative Decoding / Saguaro (Kumar, Dao, May, 2026).** Пока идёт верификация, драфтер предугадывает её вероятные исходы и заранее готовит продолжения под каждую ветку. Если реальный исход попал в подготовленный набор, стоимость драфтинга обнуляется полностью. Отличие от PEARL — не просто параллелизм, а спекуляция о результате спекуляции.

**SpecForge.** Открытый фреймворк для обучения драфтеров EAGLE-3. Упомянуть его полезно потому, что главный практический барьер сегодня — не алгоритм, а отсутствие готового чекпойнта драфтера под конкретную целевую модель.

---

## 9. За пределами уровня токенов

Если главу нужно завершить перспективой, стоит показать, что схема «черновик плюс верификация» уже вышла за пределы токенов. В работах SpecReason и SCoT спекуляция ведётся шагами рассуждения: маленькая модель предлагает промежуточный шаг цепочки размышлений, а большая его принимает или переписывает — и здесь верификация принципиально нестрогая, поскольку эквивалентных формулировок шага много. Параллельно существует линия работ по мультимодальным моделям, VLA-моделям в робототехнике и распознаванию речи, где критерий приёма ослабляется под специфику домена (например, в VLA допускается расхождение в непрерывных координатах действия в пределах допуска).

---

## 10. Сводная таблица

| Метод | Источник черновика | Структура | Lossless | Нужно обучение |
|---|---|---|---|---|
| Blockwise (Stern, 2018) | доп. головы | цепочка | да (greedy) | да |
| SpecDec / Aggressive | вход | цепочка | да | да |
| LLMA | reference-текст | цепочка | да | нет |
| Prompt Lookup | промпт | цепочка | да | нет |
| REST | внешний датастор | дерево | да | нет |
| Suffix Decoding | история выходов | дерево | да | нет |
| Token Recycling | top-k таргета | дерево | да | нет |
| Draft & Verify | пропуск слоёв | цепочка | да | нет |
| LayerSkip | ранний выход | цепочка | да | да (претрейн) |
| Kangaroo | подсеть + адаптер | цепочка | да | да (адаптер) |
| Lookahead | итерация Якоби | решётка | да | нет |
| CLLM | итерация Якоби | решётка | да | да |
| MTP | доп. головы | цепочка/дерево | да | да (претрейн) |
| DistillSpec | отдельная модель | цепочка | да/опц. нет | да |
| Online SD | отдельная модель | цепочка | да | да (онлайн) |
| CS Drafting | каскад моделей | цепочка | да | нет |
| SpecInfer | ансамбль моделей | дерево | да | да |
| Sequoia | любой | опт. дерево | да | зависит |
| SpecExec | любой | большое дерево | да | зависит |
| BiLD | отдельная модель | цепочка | **нет** | нет |
| Judge Decoding | любой | любая | **нет** | да (судья) |

---

## 11. Рекомендуемый порядок изложения

Логика, при которой материал читается как связный сюжет, а не как каталог:

1. Базовый алгоритм и его математика (правило приёма, ожидаемое число принятых токенов, оптимальная длина черновика).
2. Предыстория: Stern 2018 и SpecDec — чтобы показать, что идея старше формализации.
3. Деревья и верификация: SpecInfer, затем Sequoia и Block Verification — **до** Medusa, чтобы tree attention не выглядело её изобретением.
4. Головы поверх модели: Medusa, Hydra, EAGLE, EAGLE-2/3, MTP.
5. Self-drafting: Draft & Verify, LayerSkip, Kangaroo; отдельно Lookahead и CLLM как другой принцип.
6. Драфтинг без модели: Prompt Lookup, LLMA, REST, Suffix Decoding.
7. Выравнивание драфтера: DistillSpec, Online SD, каскады.
8. Lossy-ветка: BiLD, typical acceptance, Judge Decoding.
9. Системный слой: зависимость от батча, длинный контекст, Spec-Bench.
10. Перспектива: спекуляция на уровне рассуждений и в мультимодальных моделях.